# 07 — Dự báo phiên tiếp theo

Tương ứng src/predict.py. Kiểm tra phiên đầu vào, xem metadata mô hình, tạo dự báo và hiểu hai đầu ra độc lập.

**Cách chạy:** chọn kernel Python rồi chạy từng ô từ trên xuống (Shift+Enter), hoặc chọn **Run All**.

**Cần có trước:** Mô hình đã huấn luyện từ notebook 06 hoặc artifact được đóng gói; dùng thư viện cùng môi trường huấn luyện khi nạp joblib.

Các ô gọi hàm thật trong `src/`, không gọi `main()` nhận tham số của kernel và không chép lại thuật toán. Mỗi lần chạy có thể cập nhật tệp kết quả trong `data/processed/`, `reports/`, `figures/`, `models/`. Dữ liệu nghiên cứu dừng tại 30/09/2025.

Xem [hướng dẫn](README.md) nếu chạy trên Google Colab hoặc gặp lỗi kernel.

## 1. Tìm thư mục dự án

Chạy ô này trước các ô còn lại. Trên Colab, nếu chưa có dự án thì ô sẽ clone repo công khai; trên máy cá nhân, mở notebook từ bản clone đầy đủ.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Notebook không có __file__: tìm dự án từ thư mục đang mở và các thư mục cha.
IS_COLAB = "google.colab" in sys.modules
def is_project(folder):
    return (folder / "src" / "train.py").is_file() and (folder / "data" / "raw" / "fpt" / "fpt_historical_price.csv").is_file()

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((folder for folder in candidates if is_project(folder)), None)
if ROOT is None and IS_COLAB:
    # Colab chỉ có notebook; tải cả repo để có mã nguồn và dữ liệu đi kèm.
    clone_dir = Path("/content/fpt-stock-trends-prediction")
    if not clone_dir.exists():
        subprocess.check_call(["git", "clone", "https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git", str(clone_dir)])
    if is_project(clone_dir):
        ROOT = clone_dir
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone đầy đủ của dự án; xem notebooks/README.md.")
ROOT = ROOT.resolve()
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT)
print("Python của kernel:", sys.executable)

## 2. Kiểm tra môi trường

Local: cài `python -m pip install -r requirements-notebooks.txt` trước khi mở notebook. Colab: ô này cài phụ thuộc khi chạy. Nếu trình cài yêu cầu khởi động lại runtime, khởi động lại rồi chạy lại từ ô đầu.

In [ ]:
# Local: cài requirements-notebooks.txt trước khi mở Jupyter, để cờ False.
# Colab: mặc định cài thư viện xử lý từ requirements.txt, dùng CPU là đủ.
INSTALL_DEPENDENCIES = IS_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])
else:
    print("Dùng các thư viện đã cài trong kernel hiện tại.")

## 3. Đường dẫn và cách xem kết quả

Giá có đơn vị nghìn VND/cổ phiếu; khối lượng là số cổ phiếu. Đường dẫn là biến Python để có thể xem ngay nơi đầu vào/đầu ra được lưu.

In [ ]:
import json
import io
from contextlib import redirect_stdout
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown

# Mọi notebook dùng cùng cutoff và cùng nơi lưu kết quả như src/*.py.
CUTOFF = "2025-09-30"
PRICE_PATH = ROOT / "data/raw/fpt/fpt_historical_price.csv"
RAW_SUPPLEMENTAL_DIR = ROOT / "data/raw/supplemental"
PROCESSED_DIR = ROOT / "data/processed"
SUPPLEMENTAL_PATH = PROCESSED_DIR / "fpt_multisource_features.csv"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = ROOT / "figures"
MODELS_DIR = ROOT / "models"
for folder in (PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def run_with_log(function, *args, **kwargs):
    # Hàm gốc in JSON dài. Lưu nhật ký đầy đủ, bên dưới xem bảng dễ đọc hơn.
    log = io.StringIO()
    try:
        with redirect_stdout(log):
            result = function(*args, **kwargs)
    finally:
        log_path = ROOT / "tmp/notebook_logs" / f"{function.__name__}.txt"
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(log.getvalue(), encoding="utf-8")
    print("Nhật ký:", log_path.relative_to(ROOT))
    return result

def show_figure(name, width=1000):
    path = FIGURES_DIR / name
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có {name}; chạy ô tạo biểu đồ hoặc notebook bước trước.")
    display(Image(filename=str(path), width=width))

## 4. Xem phiên mới nhất của snapshot

Đây là giá lịch sử cuối 30/09/2025, không phải giá hôm nay. Không tự suy ra ngày t+1 bằng cách cộng một ngày vì có cuối tuần/ngày nghỉ.

In [ ]:
from features import load_price_data
prices = load_price_data(PRICE_PATH)
display(prices.tail(5))
print("Phiên cuối có dữ liệu:", prices["time"].max().date())

## 5. Kiểm tra mô hình cần những đầu vào nào

Tên biến và thứ tự phải đúng như khi học. Hàm predict_latest sẽ từ chối nếu phiên mới nhất thiếu đầu vào, không lùi về phiên cũ rồi coi như dự báo mới.

In [ ]:
import joblib
for filename in ["best_classifier.joblib", "best_regressor.joblib"]:
    path = MODELS_DIR / filename
    if not path.is_file():
        raise FileNotFoundError("Chạy notebook 06 trước: " + filename)
    artifact = joblib.load(path)
    display(Markdown(f"### {filename}: {artifact['model_name']}"))
    display(pd.Series({key: value for key, value in artifact.items() if key != "model"}).to_frame("Metadata"))

## 6. Tạo và lưu dự báo

Phân loại xuất up/down và xác suất lớp up trong hai lớp tăng/giảm; hồi quy xuất một mức giá. Hai mô hình có thể cho chiều không trùng nhau vì tối ưu mục tiêu khác nhau.

In [ ]:
from predict import predict_latest
needed_models = [MODELS_DIR / "best_classifier.joblib", MODELS_DIR / "best_regressor.joblib"]
if not all(path.is_file() for path in needed_models):
    raise FileNotFoundError("Chưa có mô hình: chạy bước train ở 00_RUN_ALL hoặc notebook 06 trước.")
prediction = run_with_log(predict_latest, PRICE_PATH, MODELS_DIR, REPORTS_DIR / "latest_prediction.json", SUPPLEMENTAL_PATH)
display(pd.Series(prediction).to_frame("Dự báo"))

## 7. Đọc kết quả bằng tiếng Việt

Xác suất 0,53 nghĩa là mô hình cho lớp Tăng khoảng 53% trong hai lớp, không phải cam kết lợi nhuận. Last-Close dự báo đứng giá; bảng này chỉ minh họa học thuật sau ngày snapshot.

In [ ]:
direction_vi = {"up": "Tăng", "down": "Giảm"}[prediction["predicted_direction"]]
display(pd.DataFrame([{
    "Dữ liệu đến phiên": prediction["as_of_session"],
    "Hướng dự báo": direction_vi,
    "Xác suất lớp Tăng (%)": 100 * prediction["probability_up_conditional_on_nonflat_direction"],
    "Close hiện tại (nghìn VND)": prediction["current_close"],
    "Giá dự báo (nghìn VND)": prediction["predicted_next_close"],
    "Mức thay đổi giá dự báo (%)": prediction["predicted_change_percent_from_current"],
}]).round(4))
print("Đã lưu reports/latest_prediction.json")